<a href="https://colab.research.google.com/github/JasmineLee7/movie-rec/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Milestone 4: What Did It Learn, and What Would It Recommend to You?

#Part 1: Set up

In [ ]:
import os
import subprocess
import sys

USER = 'JasmineLee7'
REPO = 'movie-rec'

if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', f'https://github.com/{USER}/{REPO}.git', REPO],
        check=True)

sys.path.insert(0, f'{REPO}/src')
from evaluate import to_matrix
from models import baseline_matrix, rank_k

In [ ]:
import urllib.request
import zipfile
import numpy as np
import pandas as pd

if not os.path.exists('ml-100k'):
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
        'ml-100k.zip')
    zipfile.ZipFile('ml-100k.zip').extractall()

ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t',
    names=['user', 'item', 'rating', 'time'])

In [ ]:
titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[1])[1].to_numpy()
print(len(titles), titles[49])

1682 Star Wars (1977)


#Part 2 Fit Model 2 on all the ratings

In [ ]:
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

This builds the full rating matrix using all available ratings (rather than holding out a test set like in Milestone 3), computes the SVD on the baseline-subtracted residuals, and creates a boolean mask for the 603 "popular" movies that have at least 50 ratings so obscure movies with only a few ratings don't skew the results.

# Part 3: What did it learn? Naming the hidden factors

In [ ]:
def show_factor(f, n=6):
    """Popular movies at the two ends of hidden factor f (f = 1, 2, ...)."""
    v = Vt[f - 1]
    idx = np.where(popular)[0]
    order = idx[np.argsort(v[idx])]
    print(f"Factor {f}, one end:")
    for m in order[:n]:
        print(f" {v[m]:+.3f} {titles[m]}")
    print(f"Factor {f}, other end:")
    for m in order[::-1][:n]:
        print(f" {v[m]:+.3f} {titles[m]}")

show_factor(1)

This function looks at one hidden factor, represented by a row of `Vt`. Since Python starts counting at 0, factor `f` is row `f - 1`. It keeps only movies with at least 50 ratings, sorts their factor scores, and prints the six lowest and six highest scores with their titles. These scores are factor coordinates, not predicted star ratings. Comparing the two groups helps us describe the preference pattern that the SVD found.

**Factor 1: Dark or unconventional films vs. mainstream crowd-pleasers.** The interpretation below follows the assignment's reference output; run Parts 1-3 to confirm the movie groups in this notebook. Films such as *A Clockwork Orange*, *Trainspotting*, and *Pulp Fiction* appear together at one end, while *Independence Day*, *Return of the Jedi*, and *Back to the Future* appear at the other. A reasonable interpretation is that this factor separates darker, more unconventional films from mainstream adventures and crowd favorites. This is our description of the pattern, not a genre label given to the model: the SVD used rating patterns only. The signs can flip between SVD runs, so the groups matter more than which side is positive.

In [ ]:
show_factor(3)

This calls the same function for factor 3, using the SVD already calculated in Part 2. It prints popular movies at the two ends of the third factor so we can compare a different preference pattern without fitting the model again.

**Factor 3: Science fiction and action vs. musicals and family films.** The interpretation below follows the assignment's reference output; run Parts 1-3 to confirm the movie groups in this notebook. Movies such as *Alien*, *Aliens*, and *Star Trek: First Contact* appear at one end, while *The Sound of Music*, *Grease*, and *Mary Poppins* appear at the other. This suggests a contrast between science fiction/action and musicals/family films. The split is not perfect: a factor can combine several patterns, and some movies do not fit a simple name. These interpretations describe associations in the ratings, not proof that every user likes or dislikes an entire genre. As with factor 1, the signs may be reversed without changing the pattern.